# Daily minimum-risk portfolio

Every day the weights `w` are chosen to minimise the forecast variance of the portfolio, `w' Σ w`, with `Σ` built from that day's forecasts:

- `D`: the model's variance forecast of every ticker (here the long model `xgb_long_drv_har`, which predicts the change of the realized variance),
- `R`: the correlation of the EWMA covariance of that day,
- `Σ = D R D`.

Constraints: the weights sum to 1 and none is negative (long only, fully invested). The weights of a day use only forecasts made before that day; the portfolio return of the day is `w · r` (the day's returns).
Nothing is trained here: the forecasts are read from the saved run.

In [1]:
import contextlib
import io

import numpy as np
import pandas as pd
from scipy.optimize import minimize

import run_store as rs

pd.set_option('display.width', 200)
TICKERS = ['AAPL', 'JPM', 'XOM', 'KO']
WF = 'from2016-03-01_to2026-06-30_train756bars_forecast21bars_AAPL0.25-JPM0.25-XOM0.25-KO0.25_1af3984c'   # walk-forward of train_macro_experiment.py
runs = {r.name: r for r in rs.load_runs(train_bars=756, forecast_bars=21) if r.walk_forward_id == WF}
bank = runs['xgb_long_drv_har'].ewma_bank       # per forecast day: the model's ticker variances (sigma_*), the EWMA covariance (cov_*), the day's returns (return_*)
bank[[f'sigma_{t}' for t in TICKERS] + [f'return_{t}' for t in TICKERS]].head()

,sigma_AAPL,sigma_JPM,sigma_XOM,sigma_KO,return_AAPL,return_JPM,return_XOM,return_KO
Date,,,,,,,,
2019-03-04,0.000069,0.000108,0.000109,0.000057,0.005029,-0.002298,0.003875,0.005950
2019-03-05,0.000135,0.000100,0.000124,0.000067,-0.001820,-0.000768,-0.001494,-0.001095
2019-03-06,0.000071,0.000078,0.000093,0.000055,-0.005754,-0.003746,-0.011348,-0.003290
2019-03-07,0.000078,0.000084,0.000115,0.000051,-0.011575,-0.007231,0.011100,-0.003740
2019-03-08,0.000092,0.000094,0.000107,0.000052,0.002377,0.000388,-0.014346,-0.009717


## 1. Σ of a day and the weights that minimise `w' Σ w`

In [2]:
def sigma_matrices(bank):
    """Sigma_d = D R D for every day: D = sqrt of the model's ticker variance, R = the correlation of that day's EWMA covariance."""
    d = np.sqrt(bank[[f'sigma_{t}' for t in TICKERS]].to_numpy())
    cov = np.stack([bank[[f'cov_{i}_{j}' for j in TICKERS]].to_numpy() for i in TICKERS], axis=1)      # day x i x j
    diag = np.sqrt(np.einsum('dii->di', cov))
    corr = cov / (diag[:, :, None] * diag[:, None, :])
    return d[:, :, None] * corr * d[:, None, :]


def min_variance_weights(sigma, start):
    """min w' S w  s.t. sum w = 1, w >= 0.  SLSQP; S is divided by its mean diagonal so that the numbers are about 1."""
    s = sigma / (np.trace(sigma) / len(sigma))
    res = minimize(lambda w: w @ s @ w, start, jac=lambda w: 2 * s @ w, bounds=[(0, 1)] * len(start),
                   constraints=[{'type': 'eq', 'fun': lambda w: w.sum() - 1, 'jac': lambda w: np.ones_like(w)}],
                   method='SLSQP', options={'ftol': 1e-12, 'maxiter': 200})
    w = np.clip(res.x, 0, None)
    return w / w.sum()


sigmas = sigma_matrices(bank)
day = pd.Timestamp('2024-03-15')
k = bank.index.get_loc(day)
vol = np.sqrt(np.diag(sigmas[k]) * 252)
corr = pd.DataFrame(sigmas[k] / np.outer(np.sqrt(np.diag(sigmas[k])), np.sqrt(np.diag(sigmas[k]))), index=TICKERS, columns=TICKERS)
print(f'{day.date()}: forecast annualised volatility per ticker (%):', {t: float(v) for t, v in zip(TICKERS, (vol * 100).round(1))})
corr.round(2)

2024-03-15: forecast annualised volatility per ticker (%): {'AAPL': 17.8, 'JPM': 19.2, 'XOM': 16.1, 'KO': 11.8}


,AAPL,JPM,XOM,KO
AAPL,1.00,-0.33,0.01,0.07
JPM,-0.33,1.00,0.08,0.22
XOM,0.01,0.08,1.00,0.04
KO,0.07,0.22,0.04,1.00


In [3]:
w_day = min_variance_weights(sigmas[k], np.full(4, 0.25))
equal = np.full(4, 0.25)
annual_vol = lambda weights: round(float(np.sqrt(weights @ sigmas[k] @ weights * 252)) * 100, 2)
print(f'forecast volatility of the portfolio (% / year): minimum risk {annual_vol(w_day)}, equal weight {annual_vol(equal)}')
pd.DataFrame({'weight (%)': (w_day * 100).round(1), 'equal weight (%)': equal * 100}, index=TICKERS)

forecast volatility of the portfolio (% / year): minimum risk 8.0, equal weight 8.17


,weight (%),equal weight (%)
AAPL,24.7,25.0
JPM,18.5,25.0
XOM,21.5,25.0
KO,35.3,25.0


## 2. Every day: the weights as percentages

In [4]:
w = np.full(4, 0.25)
rows = []
for s in sigmas:
    w = min_variance_weights(s, w)      # the solution of the day before is the starting point; the optimum is unique
    rows.append(w)
weights = pd.DataFrame(rows, index=bank.index, columns=TICKERS)
returns = bank[[f'return_{t}' for t in TICKERS]]
weights_pct = (weights * 100).round(2)
weights_pct.insert(0, 'forecast vol (% / year)', (np.sqrt(np.einsum('di,dij,dj->d', weights.to_numpy(), sigmas, weights.to_numpy()) * 252) * 100).round(2))
weights_pct['day return (%)'] = ((weights * returns.to_numpy()).sum(axis=1) * 100).round(3)
weights_pct.index.name = 'Date'
print(len(weights_pct), 'days; the first and last rows (the full table is `weights_pct`)')
weights_pct

1842 days; the first and last rows (the full table is `weights_pct`)


,forecast vol (% / year),AAPL,JPM,XOM,KO,day return (%)
Date,,,,,,
2019-03-04,8.70,32.43,7.72,10.17,49.68,0.480
2019-03-05,9.84,15.18,19.09,12.53,53.21,-0.119
2019-03-06,8.42,26.24,15.78,10.20,47.79,-0.483
2019-03-07,8.52,25.60,16.25,5.58,52.57,-0.549
2019-03-08,8.76,19.80,11.84,14.38,53.97,-0.679
...,...,...,...,...,...,...
2026-06-24,8.33,22.37,26.27,23.93,27.43,-0.533
2026-06-25,8.60,28.35,26.36,13.51,31.78,-1.609
2026-06-26,10.47,6.87,30.54,21.55,41.04,0.632


## 3. What the weights look like

In [5]:
summary = pd.DataFrame({
    'mean weight (%)': (weights.mean() * 100).round(1),
    'min weight (%)': (weights.min() * 100).round(1),
    'max weight (%)': (weights.max() * 100).round(1),
    'days with 0 weight (%)': ((weights < 1e-6).mean() * 100).round(1),
    'days with weight > 50% (%)': ((weights > 0.5).mean() * 100).round(1),
})
print('mean daily turnover (sum of |change of the weights|, % of the portfolio):', round(float(weights.diff().abs().sum(axis=1).mean() * 100), 1))
print('days with all the weight in one ticker:', int((weights > 0.999).any(axis=1).sum()))
summary

mean daily turnover (sum of |change of the weights|, % of the portfolio): 32.0
days with all the weight in one ticker: 14


,mean weight (%),min weight (%),max weight (%),days with 0 weight (%),days with weight > 50% (%)
AAPL,18.2,0.0,100.0,14.1,2.9
JPM,17.7,0.0,68.6,9.7,0.5
XOM,15.9,0.0,72.6,13.5,0.4
KO,48.2,0.0,100.0,0.9,42.7


## 4. Realized risk against the fixed equal-weight portfolio

Annualised volatility of the daily portfolio return (weights of the day applied to that day's returns), over the whole period and in two parts.

In [6]:
equal_return = returns.mean(axis=1)                                  # 25% each, rebalanced daily
minvar_return = (weights * returns.to_numpy()).sum(axis=1)


def period(mask):
    out = {}
    for name, r in (('equal weight', equal_return), ('minimum risk', minvar_return)):
        r = r[mask]
        out[name] = {
            'days': int(mask.sum()), 'volatility (% / year)': round(float(r.std() * np.sqrt(252)) * 100, 1),
            'worst day (%)': round(float(r.min()) * 100, 2), 'mean of the worst 5% days (%)': round(float(r[r <= r.quantile(0.05)].mean()) * 100, 2),
        }
    return pd.DataFrame(out).T.astype({'days': int})


idx = bank.index
pd.concat({
    'whole period': period(np.ones(len(idx), bool)),
    'Covid crash (2020-02-19 .. 2020-06-30)': period((idx >= '2020-02-19') & (idx <= '2020-06-30')),
    'after Covid (2020-07-01 on)': period(idx >= '2020-07-01'),
})

days  volatility (% / year)  worst day (%)  mean of the worst 5% days (%)
whole period                           equal weight  1842                   20.6         -10.99                          -3.02
                                       minimum risk  1842                   17.7          -9.67                          -2.59
Covid crash (2020-02-19 .. 2020-06-30) equal weight    93                   59.1         -10.99                          -8.96
                                       minimum risk    93                   49.5          -9.67                          -7.28
after Covid (2020-07-01 on)            equal weight  1506                   16.6          -6.60                          -2.25
                                       minimum risk  1506                   14.5          -5.74                          -2.01

## 5. Daily returns: minimum risk against equal weight

The realized returns are read from the walk-forward's calendars (`forecast_data` of every window: the returns of its forecast days), not from the bank. The equal-weight return is the calendar's own `portfolio_return` (25% each); the minimum-risk return applies the day's weights to the tickers' returns of that day.

In [7]:
DATA_PATH = './data_general/4_ticker_data_2010_2026.csv'
prices = pd.read_csv(DATA_PATH, parse_dates=['Date'], index_col='Date')
setting = rs.load_walk_forward(WF, prices)                       # the setting of the run's walk-forward (the hash of the data is checked)
with contextlib.redirect_stdout(io.StringIO()):                  # the calendars print a warning for every market holiday they skip
    calendars = setting.make_calendars()                         # one calendar per window
forecast_days = pd.concat([c.forecast_data for c in calendars])  # the realized returns of every forecast day
calendar_returns = forecast_days[TICKERS].loc[weights.index]
print('days', len(forecast_days), '| the calendars give the same returns as the bank:', bool(np.allclose(calendar_returns.to_numpy(), returns.to_numpy())))

equal_weight = forecast_days['portfolio_return'].loc[weights.index]                     # the calendar's own portfolio return
minimum_risk = (weights * calendar_returns).sum(axis=1)                                 # the day's weights applied to that day's returns
daily = pd.DataFrame({'equal weight (%)': equal_weight * 100, 'minimum risk (%)': minimum_risk * 100})
daily['difference (%)'] = daily['minimum risk (%)'] - daily['equal weight (%)']
daily.round(3)

days 1842 | the calendars give the same returns as the bank: True


,equal weight (%),minimum risk (%),difference (%)
Date,,,
2019-03-04,0.314,0.480,0.166
2019-03-05,-0.129,-0.119,0.010
2019-03-06,-0.603,-0.483,0.120
2019-03-07,-0.286,-0.549,-0.262
2019-03-08,-0.532,-0.679,-0.147
...,...,...,...
2026-06-24,-0.571,-0.533,0.039
2026-06-25,-1.341,-1.609,-0.268
2026-06-26,0.835,0.632,-0.203


In [8]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

REFERENCE_COLOR, MODEL_COLOR = '#9aa0a6', '#1f5fbf'          # the colours of the dashboard: the reference (grey) and the model (blue)
CRASH = ('2020-02-19', '2020-06-30')                          # the Covid crash
WINDOW = 63
rolling_volatility = lambda r: r.rolling(WINDOW).std() * np.sqrt(252) * 100

fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.07, row_heights=[0.38, 0.38, 0.24],
    subplot_titles=('Daily return (%)', 'Cumulative return (growth of 1, the weights of the day, rebalanced daily)', f'Realized volatility over the last {WINDOW} days (% / year)'),
)
for name, r, color in (('equal weight', equal_weight, REFERENCE_COLOR), ('minimum risk', minimum_risk, MODEL_COLOR)):
    fig.add_trace(go.Scatter(x=r.index, y=r * 100, name=name, legendgroup=name, line=dict(color=color, width=1)), row=1, col=1)
    fig.add_trace(go.Scatter(x=r.index, y=(1 + r).cumprod(), name=name, legendgroup=name, showlegend=False, line=dict(color=color, width=1.6)), row=2, col=1)
    fig.add_trace(go.Scatter(x=r.index, y=rolling_volatility(r), name=name, legendgroup=name, showlegend=False, line=dict(color=color, width=1.6)), row=3, col=1)

# the daily return as a log return, ln(1 + r), hidden until the button is pressed (the cumulative return and the volatility are the same either way)
n_simple = len(fig.data)
for name, r, color in (('equal weight', equal_weight, REFERENCE_COLOR), ('minimum risk', minimum_risk, MODEL_COLOR)):
    fig.add_trace(go.Scatter(x=r.index, y=np.log1p(r) * 100, name=name, legendgroup=name, visible=False, line=dict(color=color, width=1)), row=1, col=1)
simple_view = [True] * n_simple + [False, False]
log_view = [i not in (0, 3) for i in range(n_simple)] + [True, True]      # traces 0 and 3 are the simple daily returns

fig.add_vrect(x0=CRASH[0], x1=CRASH[1], fillcolor='#ef6c00', opacity=0.10, line_width=0, row='all', col=1)
fig.update_yaxes(ticksuffix='%', row=1, col=1)
fig.update_yaxes(ticksuffix='%', row=3, col=1)
fig.update_xaxes(rangeslider=dict(visible=True, thickness=0.05), row=3, col=1)
fig.update_layout(
    template='plotly_white', hovermode='x unified', height=860,
    legend=dict(orientation='h', yanchor='bottom', y=1.04, xanchor='right', x=1),
    updatemenus=[
        dict(type='buttons', direction='right', x=0, y=1.10, xanchor='left', yanchor='bottom', buttons=[
            dict(label='daily return: percent', method='restyle', args=[{'visible': simple_view}]),
            dict(label='daily return: log, ln(1 + r)', method='restyle', args=[{'visible': log_view}]),
        ]),
        dict(type='buttons', direction='right', x=0.50, y=1.10, xanchor='left', yanchor='bottom', buttons=[
            dict(label='growth of 1: linear', method='relayout', args=[{'yaxis2.type': 'linear'}]),
            dict(label='growth of 1: log axis', method='relayout', args=[{'yaxis2.type': 'log'}]),
        ]),
    ],
)
fig

## Caveats seen in this prototype

- The weights move a lot from day to day (the turnover above): there is no cost for trading and no limit on the change.
- There is no cap on a weight, so the portfolio often sits in the quietest ticker (KO).
- The optimiser looks for the portfolio the forecast covariance says is the safest, so it picks the places where the forecast under-estimates the risk: the realized `r²` is about 1.5 times the forecast variance of the chosen portfolio (equal weights are nearer to 1.2-1.3). A VaR computed from this `Σ` alone would be too low.